<h3 align='center'>AIRLINE</h3>
<h3 align='center'>AIRLINE OPERATIONS, DELAY & RELIABILITY ANALYTICS</h3>

<h4 align='center'>NOTEBOOK 02 - DATA ACQUISITION</h4>

**Objective:** Acquire the official BTS source data and establish reproducible raw-data provenance.

**Principle:** Raw source files are evidence. They must not be modified during acquisition.

## Imports

In [1]:
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
from datetime import datetime

## Project Paths

In [ ]:
PROJECT_PATH = Path.cwd().parent
print("Project Path", PROJECT_PATH.resolve())

RAW_PATH = PROJECT_PATH / "01_Raw_Data"
RAW_PATH.mkdir(parents=True, exist_ok=True)
print("Raw_Path", RAW_PATH.resolve())
DOCUMENTATION_PATH = PROJECT_PATH / "02_Documentation"
DOCUMENTATION_PATH.mkdir(parents=True, exist_ok=True)
print("Documentation_Path", DOCUMENTATION_PATH.resolve())

Project Path D:\arc\Python\Python Projects\airline_operations_analytics
Raw_Path D:\arc\Python\Python Projects\airline_operations_analytics\01_Raw_Data
Documentation_Path D:\arc\Python\Python Projects\airline_operations_analytics\02_Documentation


## Source Metadata

In [3]:
SOURCE_NAME = (
    "U.S. Department of Transportation - "
    "Bureau of Transportation Statistics (BTS)"
)

DATASET_NAME = (
    "Reporting Carrier On-Time Performance"
)

SOURCE_URL = (
    "https://transtats.bts.gov/"
)

FIELD_DEFINITIONS_URL = (
    "https://www.transtats.bts.gov/Fields.asp"
)

DOWNLOAD_DATE = datetime.now().strftime("%Y-%m-%d")

DATA_YEAR = 2024
DATA_MONTH = 1

print("Source:", SOURCE_NAME)
print("Dataset:", DATASET_NAME)
print("Year:", DATA_YEAR)
print("Month:", DATA_MONTH)
print("Acquisition Date:", DOWNLOAD_DATE)

Source: U.S. Department of Transportation - Bureau of Transportation Statistics (BTS)
Dataset: Reporting Carrier On-Time Performance
Year: 2024
Month: 1
Acquisition Date: 2026-09-17


## Raw Directory

In [21]:
YEAR_PATH = RAW_PATH / str(DATA_YEAR)

YEAR_PATH.mkdir(
    parents=True,
    exist_ok=True
)

print("Raw Year Directory:")
print(YEAR_PATH.resolve())

EXTRACT_PATH = YEAR_PATH / "January"

EXTRACT_PATH.mkdir(
    parents=True,
    exist_ok=True
    )

print("Extract_Path", EXTRACT_PATH.resolve())

Raw Year Directory:
D:\arc\Python\Python Projects\airline_operations_analytics\01_Raw_Data\2024
Extract_Path D:\arc\Python\Python Projects\airline_operations_analytics\01_Raw_Data\2024\January


## Detect Downloaded Source File

In [6]:
csv_files = list(
    YEAR_PATH.rglob("*.csv")
)

print("CSV files found:", len(csv_files))

for file in csv_files:
    print("-", file.name)

CSV files found: 1
- On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv


## Select Actual File

In [7]:
if not csv_files:
    raise FileNotFoundError(
        "No CSV source file found in the 2024 raw-data directory."
    )

RAW_FILE = csv_files[0]

print("Selected raw file:")
print(RAW_FILE.resolve())

Selected raw file:
D:\arc\Python\Python Projects\airline_operations_analytics\01_Raw_Data\2024\January\On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv


## Read Raw File

In [10]:
df_raw = pd.read_csv(
    RAW_FILE,
    low_memory=False
)

print("Raw dataset loaded successfully.")
print("Rows:", f"{len(df_raw):,}")
print("Columns:", len(df_raw. columns))

Raw dataset loaded successfully.
Rows: 547,271
Columns: 110


## Acquisition Validation

In [12]:
assert len(df_raw) > 0
assert len(df_raw.columns) > 0

print("Dataset contains records : PASS")
print("Dataset contains columns : PASS")

Dataset contains records : PASS
Dataset contains columns : PASS


## Raw Schema Capture

In [14]:
raw_schema = pd. DataFrame({
    "Column_Position": range(1, len(df_raw.columns) + 1),
    "Column_Name": df_raw.columns,
    "Data_Type": df_raw.dtypes.astype(str) .values
})

display(raw_schema)

,Column_Position,Column_Name,Data_Type
0,1,Year,int64
1,2,Quarter,int64
2,3,Month,int64
3,4,DayofMonth,int64
4,5,DayOfWeek,int64
...,...,...,...
105,106,Div5TotalGTime,float64
106,107,Div5LongestGTime,float64
107,108,Div5WheelsOff,float64
108,109,Div5TailNum,float64


## Raw File Statistics

In [15]:
raw_statistics = pd. DataFrame({
    "Metric": [
        "File Name",
        "File Size (MB)",
        "Rows",
        "Columns",
        "Total Cells",
        "Download Date",
        "Source Year",
        "Source Month"
    ],

    "Value": [
        RAW_FILE.name,
        round(RAW_FILE.stat().st_size / (1024 ** 2), 2),
        len(df_raw),
        len(df_raw. columns),
        len(df_raw) * len(df_raw.columns),
        DOWNLOAD_DATE,
        DATA_YEAR,
        DATA_MONTH
    ]
})

display(raw_statistics)

,Metric,Value
0,File Name,On_Time_Reporting_Carrier_On_Time_Performance_...
1,File Size (MB),235.4
2,Rows,547271
3,Columns,110
4,Total Cells,60199810
5,Download Date,2026-09-17
6,Source Year,2024
7,Source Month,1


## Raw Data Inventory

In [16]:
raw_inventory = pd. DataFrame({
    "Source": [SOURCE_NAME],
    "Dataset": [DATASET_NAME],
    "Year": [DATA_YEAR],
    "Month": [DATA_MONTH],
    "File_Name": [RAW_FILE.name],
    "Rows": [len(df_raw)],
    "Columns": [len(df_raw.columns)],
    "File_Size_MB": [
        round(
            RAW_FILE.stat().st_size / (1024 ** 2),
            2
        )
    ],
    "Acquisition_Date": [DOWNLOAD_DATE]
})

display(raw_inventory)

,Source,Dataset,Year,Month,File_Name,Rows,Columns,File_Size_MB,Acquisition_Date
0,U.S. Department of Transportation - Bureau of ...,Reporting Carrier On-Time Performance,2024,1,On_Time_Reporting_Carrier_On_Time_Performance_...,547271,110,235.4,2026-09-17


## Save Acquisition Inventory

In [18]:
inventory_file = (
    DOCUMENTATION_PATH /
    "02_Raw_Data_Inventory.xlsx"
)

with pd.ExcelWriter(
    inventory_file,
    engine="openpyxl"
) as writer:

    raw_inventory. to_excel(
        writer,
        sheet_name="Inventory",
        index=False
    )

    raw_schema. to_excel(
        writer,
        sheet_name="Raw_Schema",
        index=False
    )

    raw_schema.to_excel(
        writer,
        sheet_name="File_Statistics",
        index=False
    )

print("Acquisition inventory saved:")
print(inventory_file.resolve())

Acquisition inventory saved:
D:\arc\Python\Python Projects\airline_operations_analytics\02_Documentation\02_Raw_Data_Inventory.xlsx


## Acquisiton Audit

In [19]:
acquisition_audit = pd.DataFrame({
    "Control": [
        "Official source identified",
        "Raw file located",
        "Raw file successfully loaded",
        "Row count captured",
        "Column count captured",
        "Raw schema captured",
        "Acquisition metadata captured"
    ],

    "Status": [
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS",
        "PASS"
    ]
})

display(acquisition_audit)

,Control,Status
0,Official source identified,PASS
1,Raw file located,PASS
2,Raw file successfully loaded,PASS
3,Row count captured,PASS
4,Column count captured,PASS
5,Raw schema captured,PASS
6,Acquisition metadata captured,PASS


In [20]:
print("=" * 60)
print("AIRLINE - DATA ACQUISITION GATE")
print("=" * 60)

print(f"Source File : {RAW_FILE.name}")
print(f"Rows        : {len(df_raw):,}")
print(f"Columns     : {len(df_raw.columns):,}")
print(f"Source Year : {DATA_YEAR}")
print(f"Source Month: {DATA_MONTH}")

print("=" * 60)
print("DATA ACQUISITION COMPLETE")
print("NEXT PHASE → DATA INVENTORY / DATA UNDERSTANDING")
print("=" * 60)

AIRLINE - DATA ACQUISITION GATE
Source File : On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2024_1.csv
Rows        : 547,271
Columns     : 110
Source Year : 2024
Source Month: 1
DATA ACQUISITION COMPLETE
NEXT PHASE → DATA INVENTORY / DATA UNDERSTANDING


In [17]:
flight_dates = pd.to_datetime(
    df["FlightDate"].astype("str"),
    format= "%Y-%m-%d",
    errors= "coerce"
)

print("Min Flight Dates: ", min(flight_dates))
print("Max Flight Dates: ", max(flight_dates))

Min Flight Dates:  2024-01-01 00:00:00
Max Flight Dates:  2024-01-31 00:00:00
